# Why Every Transformer Moved the LayerNorm

**Topic:** Pre-norm vs post-norm &nbsp;·&nbsp; **Difficulty:** Hard

The original transformer put LayerNorm *after* the residual addition. Every model
since about 2020 puts it *before* the sublayer instead, and the reason is
visible in one measurement: the gradient reaching layer 1 of a 48-layer stack.

Pre-norm leaves a clean identity path from the loss all the way to block 1, so
the gradient there is *larger* than the one at the output, because the residual
contributions accumulate. Post-norm puts a LayerNorm on that path, and its
Jacobian attenuates the gradient at every one of the 48 steps. Measure the two
and compare.

## The data (created in the next cell)

- A stack of `L = 48` blocks, each `x -> x + W_2 * relu(W_1 * f(x))` where `f` is
  either `layernorm` (pre-norm) or the identity with the norm applied after the
  addition (post-norm).
- Width 64, weights seeded with `np.random.default_rng(21)`.
- Input `x0` `(32, 64)`.

## Your task

With **NumPy only**:

1. Implement `layernorm(x)` over the feature axis (`eps = 1e-5`, biased
   variance as `np.var` gives, no learned scale or shift).
2. Implement both stacks and, for each, backpropagate a fixed upstream gradient
   `g_out` of ones through all 48 blocks, recording `||g||_F` at the input of
   every block.
3. Set `answers` to a dict with `"grad_pre"`, `"grad_post"` (length-48 arrays of
   gradient norms, block 1 first), `"ratio_pre"`, `"ratio_post"`: the norm at
   block 1 divided by the norm at block 48 for each.

## Evaluation

Compared to the reference using a tolerance of `1e-6 * (1 + |reference|)`,
relative for large values and an absolute `1e-6` floor near zero. The evaluation
cell checks the pre-norm stack delivers at least 1000 times more gradient to
block 1 than the post-norm stack does. The post-norm curve is rounding noise
(about 1e-15 per block), so it is only checked to lie within that absolute floor
of zero, and `ratio_post`, a ratio of two noise values, is not graded.

In [ ]:
import numpy as np

rng = np.random.default_rng(21)
L, d, n = 48, 64, 32
W1 = rng.normal(0, np.sqrt(2.0 / d), (L, d, d))
W2 = rng.normal(0, np.sqrt(2.0 / d), (L, d, d))
x0 = rng.normal(0, 1, (n, d))
eps = 1e-5
print(f"{L} blocks of width {d}, input {x0.shape}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the data created above to compute the gradient norms, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("grad_pre", [48], 4078.361516899614, 8.098394860649393, 1e-06),
    ("grad_post", [48], 1.4677358459105427e-13, -1.3670957065385139e-15, 1e-06),
    ("ratio_pre", [], 10.334509626055931, -1.7303897758681372, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "grad_pre" in answers and "grad_post" in answers:
    _g0 = float(answers["grad_post"][0])
    _decay = _g0 >= 0.0 and float(answers["grad_pre"][0]) > 1000.0 * _g0
    print(f"  {'block 1: pre >> post':<24} {'ok' if _decay else 'wrong'}  "
          f"(pre {float(answers['grad_pre'][0]):.3g}, post {_g0:.3g})")
    _ok = _ok and _decay

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

In the pre-norm stack the residual is `x + F(LN(x))`, so `dL/dx` picks up an
additive `1` at every block: the gradient reaching block 1 is bounded below by the
gradient at the output, regardless of depth. In post-norm the block is
`LN(x + F(x))`, so every backward step also multiplies by LayerNorm's Jacobian,
which projects out the mean and rescales by `1/std`. With random weights at this depth the post-norm gradient at block 1 is already at
rounding-error scale: the upstream gradient of all ones is exactly the constant
direction LayerNorm subtracts away, so the first backward pass through the norm
leaves only floating-point noise. Pre-norm's identity path keeps the full gradient
flowing, so its block-1 norm sits many orders of magnitude above post-norm's. The
block-1-to-block-48 gradient-norm ratio is about 10 for pre-norm and noise-level
for post-norm, and the gap widens with depth and with trained weights, which is
why post-norm needs a warmup schedule and pre-norm largely does not.

```python
import numpy as np

def layernorm(x):
    mu = x.mean(axis=-1, keepdims=True)
    var = x.var(axis=-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps), mu, var

def ln_backward(g, x, mu, var):
    d_ = x.shape[-1]
    std = np.sqrt(var + eps)
    xh = (x - mu) / std
    gxh = g
    return (gxh - gxh.mean(axis=-1, keepdims=True)
            - xh * (gxh * xh).mean(axis=-1, keepdims=True)) / std

def run_pre():
    xs, cache = [], []
    x = x0.copy()
    for i in range(L):
        xn, mu, var = layernorm(x)
        h = np.maximum(xn @ W1[i], 0.0)
        out = x + h @ W2[i]
        cache.append((x, xn, mu, var, h))
        xs.append(out)
        x = out
    g = np.ones_like(x)
    norms = []
    for i in reversed(range(L)):
        xin, xn, mu, var, h = cache[i]
        gh = (g @ W2[i].T) * (h > 0)
        gxn = gh @ W1[i].T
        g = g + ln_backward(gxn, xin, mu, var)
        norms.append(float(np.linalg.norm(g)))
    return np.array(norms[::-1])

def run_post():
    cache = []
    x = x0.copy()
    for i in range(L):
        h = np.maximum(x @ W1[i], 0.0)
        s = x + h @ W2[i]
        out, mu, var = layernorm(s)
        cache.append((x, h, s, mu, var))
        x = out
    g = np.ones_like(x)
    norms = []
    for i in reversed(range(L)):
        xin, h, s, mu, var = cache[i]
        g = ln_backward(g, s, mu, var)
        gh = (g @ W2[i].T) * (h > 0)
        g = g + gh @ W1[i].T
        norms.append(float(np.linalg.norm(g)))
    return np.array(norms[::-1])

grad_pre = run_pre()
grad_post = run_post()

answers = {
    "grad_pre": grad_pre,
    "grad_post": grad_post,
    "ratio_pre": float(grad_pre[0] / grad_pre[-1]),
    "ratio_post": float(grad_post[0] / grad_post[-1]),
}
```